<a href="https://colab.research.google.com/github/3160664535qqcom-ctrl/kangtai-AnalyticsEngineering/blob/3160664535qqcom-ctrl-personal-exercise/bank_DPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import glob
import zipfile
import urllib.request
import pandas as pd
import duckdb

DATA_DIR = "/content/data"
# GitHub 归档下载 URL
DATA_URL = "https://github.com/3160664535qqcom-ctrl/kangtai-AnalyticsEngineering/archive/refs/heads/3160664535qqcom-ctrl-personal-exercise.zip"

# 1. 下载并解压主仓库数据
os.makedirs(DATA_DIR, exist_ok=True)
zip_path = "/content/kangtai-AnalyticsEngineering-3160664535qqcom-ctrl-personal-exercise.zip"

if not glob.glob(os.path.join(DATA_DIR, "**", "*.xlsx"), recursive=True):
    print("Downloading dataset...")
    try:
        urllib.request.urlretrieve(DATA_URL, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("Download and main extraction complete.")
    except Exception as e:
        print(f"Failed to download or unzip main archive: {e}")

# 2. 核心修复：检查并解压嵌套的 banks.zip 以释放 .xlsx 文件
banks_zip_candidates = glob.glob(os.path.join(DATA_DIR, "**", "banks.zip"), recursive=True)
if banks_zip_candidates:
    banks_zip_path = banks_zip_candidates[0]
    print(f"Found nested zip file: {banks_zip_path}. Extracting it to release XLSX files...")
    with zipfile.ZipFile(banks_zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Nested extraction complete.")

def find(filename):
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if not found:
        have = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.*"), recursive=True))
        raise FileNotFoundError(f"Missing {filename}. Files found under {DATA_DIR}: {have}.")
    return found[0]

# 3. 读取 Excel 文件（铜牌/Bronze 层原始数据映射）
try:
    bronze_tx = pd.read_excel(find("card_transactions_extract.xlsx"), dtype={"ref_txn_id": "string"})
    bronze_cu = pd.read_excel(find("customer_master_extract.xlsx"))
    bronze_fx = pd.read_excel(find("fx_rates_extract.xlsx"))
except Exception as e:
    print(f"Error reading Excel files: {e}. Attempting to install openpyxl...")
    import subprocess
    subprocess.run(["pip", "install", "openpyxl"])
    bronze_tx = pd.read_excel(find("card_transactions_extract.xlsx"), dtype={"ref_txn_id": "string"})
    bronze_cu = pd.read_excel(find("customer_master_extract.xlsx"))
    bronze_fx = pd.read_excel(find("fx_rates_extract.xlsx"))

print(f"\nBronze Tables Loaded:")
print(f"- bronze_transactions: {len(bronze_tx):,} rows")
print(f"- bronze_customers: {len(bronze_cu):,} rows")
print(f"- bronze_fx_rates: {len(bronze_fx):,} rows")

# 4. 初始化 DuckDB 并注册 Bronze 临时表
con = duckdb.connect()
tables = {
    "bronze_transactions": bronze_tx,
    "bronze_customers": bronze_cu,
    "bronze_fx_rates": bronze_fx
}

for name, df in tables.items():
    con.register(name, df)

def q(sql):
    """在 DuckDB 中执行 SQL 查询并返回 pandas DataFrame"""
    return con.sql(sql).df()

# 打印 DuckDB 中的 Bronze 表信息
print("\nRegistered DuckDB Bronze Tables:")
for name in tables.keys():
    info_df = q(f"DESCRIBE {name}")
    print(f"\nTable: {name}")
    display(info_df[["column_name", "column_type"]])

Download and main extraction complete.
Found nested zip file: /content/data/kangtai-AnalyticsEngineering-3160664535qqcom-ctrl-personal-exercise/banks.zip. Extracting it to release XLSX files...
Nested extraction complete.

Bronze Tables Loaded:
- bronze_transactions: 16 rows
- bronze_customers: 10 rows
- bronze_fx_rates: 16 rows

Registered DuckDB Bronze Tables:

Table: bronze_transactions


,column_name,column_type
0,txn_id,VARCHAR
1,ref_txn_id,VARCHAR
2,card_id,VARCHAR
3,merchant,VARCHAR
4,amount,DOUBLE
5,currency,VARCHAR
6,status,VARCHAR
7,txn_timestamp,VARCHAR
8,file_date,TIMESTAMP_NS



Table: bronze_customers


,column_name,column_type
0,snapshot_date,TIMESTAMP_NS
1,customer_id,VARCHAR
2,account_id,VARCHAR
3,card_id,VARCHAR
4,segment,VARCHAR
5,account_status,VARCHAR
6,open_date,TIMESTAMP_NS



Table: bronze_fx_rates


,column_name,column_type
0,rate_date,TIMESTAMP_NS
1,currency,VARCHAR
2,rate_to_gbp,DOUBLE


In [8]:
print("--- bronze_transactions data ---")
display(q("SELECT * FROM bronze_transactions"))

print("\n--- bronze_customers data ---")
display(q("SELECT * FROM bronze_customers"))

print("\n--- bronze_fx_rates data ---")
display(q("SELECT * FROM bronze_fx_rates"))

--- bronze_transactions data ---


,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
0,T1001,None,C1001,TESCO STORES 3021,23.40,GBP,PENDING,2026-03-02 09:14:22,2026-03-02
1,T1010,None,C1004,BOOKING.COM,210.00,EUR,PENDING,2026-02-28 19:55:42,2026-03-02
2,T1001,None,C1001,TESCO STORES 3021,23.40,GBP,POSTED,2026-03-02 09:14:22,2026-03-03
3,T1002,None,C1002,Tesco Stores 3021,8.75,GBP,POSTED,2026-03-02 17:40:05,2026-03-03
4,T1010,None,C1004,BOOKING.COM,210.00,EUR,POSTED,2026-02-28 19:55:42,2026-03-03
5,T1003,None,C1003,AMAZON MKTPLACE,54.99,GBP,POSTED,2026-03-03 11:02:48,2026-03-04
6,T1003,None,C1003,AMAZON MKTPLACE,54.99,GBP,POSTED,2026-03-03 11:02:48,2026-03-04
7,T1004,None,C1004,AIRBNB * HM4K2,120.00,EUR,PENDING,2026-03-03 20:31:09,2026-03-04
8,T1004,None,C1004,AIRBNB * HM4K2,120.00,EUR,POSTED,2026-03-03 20:31:09,2026-03-05
9,T1005,None,C1001,PRET A MANGER 112,6.20,gbp,POSTED,2026-03-04 08:05:51,2026-03-05



--- bronze_customers data ---


,snapshot_date,customer_id,account_id,card_id,segment,account_status,open_date
0,2026-03-02,CU001,A-7001,C1001,Standard,ACTIVE,2022-05-14
1,2026-03-02,CU002,A-7002,C1002,Standard,ACTIVE,2021-11-03
2,2026-03-02,CU003,A-7003,C1003,premium,ACTIVE,2019-08-21
3,2026-03-02,CU004,A-7004,C1004,Premium,ACTIVE,2023-01-09
4,2026-03-02,CU005,A-7005,C1005,Standard,ACTIVE,2020-02-27
5,2026-03-09,CU001,A-7001,C1001,Standard,ACTIVE,2022-05-14
6,2026-03-09,CU002,A-7002,C1002,Premium,ACTIVE,2021-11-03
7,2026-03-09,CU003,A-7003,C1003,Premium,ACTIVE,2019-08-21
8,2026-03-09,CU004,A-7004,C1004,Premium,ACTIVE,2023-01-09
9,2026-03-09,CU005,A-7005,C1005,Standard,CLOSED,2020-02-27



--- bronze_fx_rates data ---


,rate_date,currency,rate_to_gbp
0,2026-02-27,EUR,0.8512
1,2026-02-27,USD,0.7894
2,2026-03-02,EUR,0.8498
3,2026-03-02,USD,0.7881
4,2026-03-03,EUR,0.8503
5,2026-03-03,USD,0.7869
6,2026-03-04,EUR,0.8521
7,2026-03-04,USD,0.7902
8,2026-03-05,EUR,0.8490
9,2026-03-05,USD,0.7888


In [9]:
import re
import pandas as pd
import numpy as np

# Status priority mapping for keeping the latest status
STATUS_RANK = {
    "PENDING": 1,
    "POSTED": 2,
    "REVERSED": 3
}

# first steps of cleaning:

def standardise_currency_codes(df, col="currency"):
    """Trim spaces and upper-case currency codes ('gbp ' becomes 'GBP')."""
    out = df.copy()
    out[col] = out[col].str.strip().str.upper()
    return out


def standardise_merchant_text(df):
    """Tidy merchant names: trim, collapse repeated spaces, upper-case."""
    out = df.copy()
    # Convert to string, trim, uppercase, and replace multiple spaces with a single space
    out["merchant_clean"] = out["merchant"].astype(str).str.strip().str.upper().apply(lambda x: re.sub(r'\s+', ' ', x))
    return out


def parse_timestamps(df):
    """Turn the text timestamp into a real datetime."""
    out = df.copy()
    s = out["txn_timestamp"].astype(str)
    day_first = s.str.contains("/", regex=False)
    ts = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns]")
    ts[day_first] = pd.to_datetime(s[day_first], format="%d/%m/%Y %H:%M", errors="coerce")
    ts[~day_first] = pd.to_datetime(s[~day_first], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    out["ts"] = ts
    return out

# second steps:

def drop_duplicates_custom(df):
    """Remove duplicate rows."""
    return df.drop_duplicates().reset_index(drop=True)


def derive_transaction_date_and_month(df):
    """Work out the transaction date and its month (YYYY-MM)."""
    out = df.copy()
    out["tdate"] = out["ts"].dt.date
    out["month"] = out["ts"].dt.to_period("M").astype(str)
    return out

# third steps:

def keep_latest_status_per_transaction(df):
    """Keep one row per transaction id: the latest known status based on file_date and STATUS_RANK."""
    out = df.copy()
    out["_rank"] = out["status"].map(STATUS_RANK).fillna(0)
    out = out.sort_values(["txn_id", "file_date", "_rank"], ascending=[True, True, True])
    out = out.groupby("txn_id").tail(1).drop(columns="_rank")
    return out.reset_index(drop=True)


def flag_reversals(df):
    """Mark rows that are reversals (a refund recorded as its own row)."""
    out = df.copy()
    out["is_reversal"] = out["status"].str.upper() == "REVERSED"
    return out


def flag_zero_amount(df):
    """Mark transactions with an amount of exactly zero."""
    out = df.copy()
    out["zero_amount"] = out["amount"] == 0
    return out


def flag_foreign_currency(df):
    """Mark transactions made in a currency other than GBP."""
    out = df.copy()
    out["is_foreign_currency"] = out["currency"].str.upper() != "GBP"
    return out

# fourth steps:

def link_reversals_to_original(df):
    """Connect each reversal row to the original purchase it cancels."""
    out = df.copy()
    out["original_found"] = False
    out["original_amount"] = np.nan

    # Map ref_txn_id to original amounts and status
    reversals = out[out["is_reversal"] & out["ref_txn_id"].notna()]
    for idx, row in reversals.iterrows():
        orig = out[out["txn_id"] == row["ref_txn_id"]]
        if not orig.empty:
            out.loc[idx, "original_found"] = True
            out.loc[idx, "original_amount"] = orig.iloc[0]["amount"]

    return out

# Pipeline execution
processed_tx = (bronze_tx
    .pipe(drop_duplicates_custom)
    .pipe(standardise_currency_codes)
    .pipe(standardise_merchant_text)
    .pipe(parse_timestamps)
    .pipe(derive_transaction_date_and_month)
    .pipe(keep_latest_status_per_transaction)
    .pipe(flag_reversals)
    .pipe(flag_zero_amount)
    .pipe(flag_foreign_currency)
    .pipe(link_reversals_to_original)
)

print("Processing complete! Cleaned transactions in Silver Layer:")
display(processed_tx.head())

Processing complete! Cleaned transactions in Silver Layer:


,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date,merchant_clean,ts,tdate,month,is_reversal,zero_amount,is_foreign_currency,original_found,original_amount
0,R2001,T1005,C1001,PRET A MANGER 112,-6.20,GBP,REVERSED,2026-03-05 10:12:00,2026-03-06,PRET A MANGER 112,2026-03-05 10:12:00,2026-03-05,2026-03,True,False,False,True,6.2
1,T1001,<NA>,C1001,TESCO STORES 3021,23.40,GBP,POSTED,2026-03-02 09:14:22,2026-03-03,TESCO STORES 3021,2026-03-02 09:14:22,2026-03-02,2026-03,False,False,False,False,NaN
2,T1002,<NA>,C1002,Tesco Stores 3021,8.75,GBP,POSTED,2026-03-02 17:40:05,2026-03-03,TESCO STORES 3021,2026-03-02 17:40:05,2026-03-02,2026-03,False,False,False,False,NaN
3,T1003,<NA>,C1003,AMAZON MKTPLACE,54.99,GBP,POSTED,2026-03-03 11:02:48,2026-03-04,AMAZON MKTPLACE,2026-03-03 11:02:48,2026-03-03,2026-03,False,False,False,False,NaN
4,T1004,<NA>,C1004,AIRBNB * HM4K2,120.00,EUR,POSTED,2026-03-03 20:31:09,2026-03-05,AIRBNB * HM4K2,2026-03-03 20:31:09,2026-03-03,2026-03,False,False,True,False,NaN


In [10]:
# clean the customer table
import pandas as pd
import numpy as np
# Safely load bronze_cu from global variables or DuckDB if missing in memory
if 'bronze_cu' not in globals():
    try:
        bronze_cu = q("SELECT * FROM bronze_customers")
    except Exception:
        # Fallback to direct excel read if DuckDB is not initialized
        bronze_cu = pd.read_excel(find("customer_master_extract.xlsx"))

# first step:

def standardise_segment_labels(df):
    """Make segment labels consistent ('premium', 'PREMIUM ' become 'Premium').

    Needs: segment
    Adds:  segment (cleaned, same column)
    """
    out = df.copy()
    out["segment"] = out["segment"].astype(str).str.strip().str.capitalize()
    return out

# second step:

def latest_customer_snapshot_per_card(df):
    """Reduce the weekly snapshots to one row per card: the most recent record.
    Decision recorded here: 'segment' means the segment NOW (latest snapshot).

    Needs: card_id, snapshot_date, customer_id, segment, account_status
    Adds:  nothing (one row per card)
    """
    out = df.copy()
    out["snapshot_date"] = pd.to_datetime(out["snapshot_date"])
    out = out.sort_values("snapshot_date", ascending=True)
    out = out.groupby("card_id").last().reset_index()
    return out

# Pipeline execution
processed_customers = (
    bronze_cu
    .pipe(standardise_segment_labels)
    .pipe(latest_customer_snapshot_per_card)
)

print("Processed Customer Table (Latest snapshot per card):")
display(processed_customers)

Processed Customer Table (Latest snapshot per card):


,card_id,snapshot_date,customer_id,account_id,segment,account_status,open_date
0,C1001,2026-03-09,CU001,A-7001,Standard,ACTIVE,2022-05-14
1,C1002,2026-03-09,CU002,A-7002,Premium,ACTIVE,2021-11-03
2,C1003,2026-03-09,CU003,A-7003,Premium,ACTIVE,2019-08-21
3,C1004,2026-03-09,CU004,A-7004,Premium,ACTIVE,2023-01-09
4,C1005,2026-03-09,CU005,A-7005,Standard,CLOSED,2020-02-27


In [11]:
# clean the fx_rate table
import pandas as pd
import numpy as np

# Safely load bronze_fx and processed_tx if missing in memory
if 'bronze_fx' not in globals():
    try:
        bronze_fx = q("SELECT * FROM bronze_fx_rates")
    except Exception:
        bronze_fx = pd.read_excel(find("fx_rates_extract.xlsx"))

if 'processed_tx' not in globals():
    # If processed_tx hasn't been defined, run the pipeline to create it
    if 'bronze_tx' not in globals():
        bronze_tx = pd.read_excel(find("card_transactions_extract.xlsx"), dtype={"ref_txn_id": "string"})
    processed_tx = (bronze_tx
        .pipe(drop_duplicates_custom)
        .pipe(standardise_currency_codes)
        .pipe(standardise_merchant_text)
        .pipe(parse_timestamps)
        .pipe(derive_transaction_date_and_month)
        .pipe(keep_latest_status_per_transaction)
        .pipe(flag_reversals)
        .pipe(flag_zero_amount)
        .pipe(flag_foreign_currency)
        .pipe(link_reversals_to_original)
    )

# first step:

def build_fx_calendar(fx, end_date):
    """Lay the rates out on a full calendar: one row per day from the first
    published rate to `end_date`, one column per currency. Days with no published rate
    (weekends, holidays) are left empty at this stage.

    Needs: rate_date, currency, rate_to_gbp
    Adds:  a table indexed by day, one column per currency
    """
    df = fx.copy()
    df["rate_date"] = pd.to_datetime(df["rate_date"])

    # Pivot the raw FX data
    pivoted = df.pivot(index="rate_date", columns="currency", values="rate_to_gbp")

    # Create complete date range
    start_date = pivoted.index.min()
    full_idx = pd.date_range(start=start_date, end=end_date, freq="D")

    # Reindex to get the empty rows on missing dates
    fx_wide = pivoted.reindex(full_idx)
    fx_wide.index.name = "tdate"
    return fx_wide

# second step:

def fill_missing_fx_rates(fx_wide):
    """Fill days with no published rate with the last published rate.
    Decision recorded here: weekends and holidays use the previous published day.

    Needs: the calendar table from build_fx_calendar
    Adds:  nothing (empty days are filled)
    """
    return fx_wide.ffill()


def add_gbp_rate(fx_wide):
    """Add GBP itself with a rate of 1.0, so GBP transactions convert like any other.

    Needs: the calendar table
    Adds:  a GBP column (all 1.0)
    """
    out = fx_wide.copy()
    out["GBP"] = 1.0
    return out

# third step:

def flag_filled_fx_rates(fx_wide_before_fill, fx_wide_after_fill):
    """Record which days used a carried-forward rate instead of a published
    one, so a user can tell a real rate from a filled one.
    """
    filled_flags = fx_wide_before_fill.isnull() & fx_wide_after_fill.notnull()
    # Rename columns to indicate flag status
    filled_flags = filled_flags.rename(columns=lambda x: f"{x}_is_filled")
    return pd.concat([fx_wide_after_fill, filled_flags], axis=1)

# Pipeline execution
max_tx_date = pd.to_datetime(processed_tx["ts"]).max()
fx_wide_empty = build_fx_calendar(bronze_fx, end_date=max_tx_date)
fx_wide_filled = fill_missing_fx_rates(fx_wide_empty)
fx_wide_with_gbp = add_gbp_rate(fx_wide_filled)
processed_fx = flag_filled_fx_rates(fx_wide_empty, fx_wide_with_gbp)

print("Processed FX Rates Table with Daily Calendar:")
display(processed_fx)

Processed FX Rates Table with Daily Calendar:


currency,EUR,USD,GBP,EUR_is_filled,GBP_is_filled,USD_is_filled
tdate,,,,,,
2026-02-27,0.8512,0.7894,1.0,False,NaN,False
2026-02-28,0.8512,0.7894,1.0,True,NaN,True
2026-03-01,0.8512,0.7894,1.0,True,NaN,True
2026-03-02,0.8498,0.7881,1.0,False,NaN,False
2026-03-03,0.8503,0.7869,1.0,False,NaN,False
2026-03-04,0.8521,0.7902,1.0,False,NaN,False
2026-03-05,0.8490,0.7888,1.0,False,NaN,False
2026-03-06,0.8476,0.7875,1.0,False,NaN,False
2026-03-07,0.8476,0.7875,1.0,True,NaN,True


In [12]:
#join tables together
import pandas as pd
import numpy as np

def add_segment_from_latest_snapshot(tx, cards):
    """Attach each transaction's customer segment (latest snapshot).
    Cards missing from the customer master get segment 'Unknown' and are flagged,
    not dropped, so totals still reconcile.
    """
    out = tx.copy()
    # Merge on card_id
    merged = out.merge(cards[['card_id', 'segment']], on='card_id', how='left')
    merged['customer_known'] = merged['segment'].notna()
    merged['segment'] = merged['segment'].fillna('Unknown')
    return merged


def add_fx_rate_to_transactions(tx, fx_wide):
    """Look up the exchange rate for each transaction: the rate on the
    transaction DATE, for the transaction's currency.
    """
    out = tx.copy()
    # Ensure dates are in datetime format for correct mapping
    out['tdate_dt'] = pd.to_datetime(out['tdate'])

    # Melt fx_wide to have (tdate, currency, rate) format
    # Keep currency columns (not the '_is_filled' ones)
    currency_cols = [col for col in fx_wide.columns if not col.endswith('_is_filled')]
    fx_melted = fx_wide[currency_cols].reset_index().melt(id_vars='tdate', var_name='currency', value_name='rate')
    fx_melted['tdate'] = pd.to_datetime(fx_melted['tdate'])

    # Merge on tdate and currency
    merged = out.merge(fx_melted, left_on=['tdate_dt', 'currency'], right_on=['tdate', 'currency'], how='left', suffixes=('', '_fx'))
    merged = merged.drop(columns=['tdate_dt', 'tdate_fx'])
    return merged


def add_gbp_amount(df):
    """Convert each amount to pounds."""
    out = df.copy()
    out['amount_gbp'] = out.apply(
        lambda r: r['amount'] * r['rate'] if pd.notna(r['rate']) else (r['amount'] if r['currency'] == 'GBP' else np.nan),
        axis=1
    )
    return out

# Pipeline execution for Consolidated Silver table
silver_consolidated = (
    processed_tx
    .pipe(add_segment_from_latest_snapshot, cards=processed_customers)
    .pipe(add_fx_rate_to_transactions, fx_wide=processed_fx)
    .pipe(add_gbp_amount)
)

print("Consolidated Silver Layer Table (Merged with FX and Customer segments):")
display(silver_consolidated.head())

Consolidated Silver Layer Table (Merged with FX and Customer segments):


,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date,merchant_clean,...,month,is_reversal,zero_amount,is_foreign_currency,original_found,original_amount,segment,customer_known,rate,amount_gbp
0,R2001,T1005,C1001,PRET A MANGER 112,-6.20,GBP,REVERSED,2026-03-05 10:12:00,2026-03-06,PRET A MANGER 112,...,2026-03,True,False,False,True,6.2,Standard,True,1.0000,-6.200
1,T1001,<NA>,C1001,TESCO STORES 3021,23.40,GBP,POSTED,2026-03-02 09:14:22,2026-03-03,TESCO STORES 3021,...,2026-03,False,False,False,False,NaN,Standard,True,1.0000,23.400
2,T1002,<NA>,C1002,Tesco Stores 3021,8.75,GBP,POSTED,2026-03-02 17:40:05,2026-03-03,TESCO STORES 3021,...,2026-03,False,False,False,False,NaN,Premium,True,1.0000,8.750
3,T1003,<NA>,C1003,AMAZON MKTPLACE,54.99,GBP,POSTED,2026-03-03 11:02:48,2026-03-04,AMAZON MKTPLACE,...,2026-03,False,False,False,False,NaN,Premium,True,1.0000,54.990
4,T1004,<NA>,C1004,AIRBNB * HM4K2,120.00,EUR,POSTED,2026-03-03 20:31:09,2026-03-05,AIRBNB * HM4K2,...,2026-03,False,False,True,False,NaN,Premium,True,0.8503,102.036


In [13]:
def check_one_row_per_transaction(df):
    """Check that txn_id is unique and never empty in the finished silver table.

    Needs: txn_id, in the finished table
    Adds:  nothing (passes silently or reports the problem)
    """
    if df['txn_id'].isnull().any():
        raise ValueError("Validation failed: 'txn_id' contains null values.")

    if not df['txn_id'].is_unique:
        duplicated_ids = df[df['txn_id'].duplicated()]['txn_id'].unique()
        raise ValueError(f"Validation failed: 'txn_id' is not unique. Duplicates found: {duplicated_ids}")

    print("Validation passed: 'txn_id' is unique and contains no null values.")

# Run validation on the consolidated silver table
check_one_row_per_transaction(silver_consolidated)

Validation passed: 'txn_id' is unique and contains no null values.


In [14]:
# Gold Layer: Customer Segment Metrics and Spend Summaries
import pandas as pd

# Filter out reversed transactions for general spend metrics
posted_tx = silver_consolidated[silver_consolidated['status'] != 'REVERSED'].copy()

# 1. Total posted spend in GBP, per customer segment, per month.
# build through getting columns of month, segment and amount_gbp
gold_segment_monthly_spend = (
    posted_tx
    .groupby(['month', 'segment'])
    .agg(total_spend_gbp=('amount_gbp', 'sum'))
    .reset_index()
)

# 2. The number and GBP value of reversals, per month.
# build through getting columns of is_reversal, month and amount (converted to GBP)
reversals_tx = silver_consolidated[silver_consolidated['is_reversal'] == True].copy()
gold_monthly_reversals = (
    reversals_tx
    .groupby('month')
    .agg(
        reversal_count=('txn_id', 'count'),
        total_reversal_value_gbp=('amount_gbp', 'sum')
    )
    .reset_index()
)

# 3. Which cards spent in a foreign currency this month, and how much (reported in GBP)
# build through getting columns of month, segment, card_id and is_foreign_currency
foreign_spend_tx = posted_tx[posted_tx['is_foreign_currency'] == True].copy()
gold_foreign_spend_by_card = (
    foreign_spend_tx
    .groupby(['month', 'card_id', 'segment'])
    .agg(foreign_spend_gbp=('amount_gbp', 'sum'))
    .reset_index()
)

# Display Gold Tables
print("--- 1. Total Posted Spend in GBP per Customer Segment per Month ---")
display(gold_segment_monthly_spend)

print("\n--- 2. Number and GBP Value of Reversals per Month ---")
display(gold_monthly_reversals)

print("\n--- 3. Foreign Currency Spend by Card per Month (in GBP) ---")
display(gold_foreign_spend_by_card)

# Export to CSV files
gold_segment_monthly_spend.to_csv('gold_segment_monthly_spend.csv', index=False)
gold_monthly_reversals.to_csv('gold_monthly_reversals.csv', index=False)
gold_foreign_spend_by_card.to_csv('gold_foreign_spend_by_card.csv', index=False)

print("\nGold tables have been successfully exported to CSV files:")
print("- gold_segment_monthly_spend.csv")
print("- gold_monthly_reversals.csv")
print("- gold_foreign_spend_by_card.csv")

--- 1. Total Posted Spend in GBP per Customer Segment per Month ---


,month,segment,total_spend_gbp
0,2026-02,Premium,178.75200
1,2026-03,Premium,207.33762
2,2026-03,Standard,41.64875
3,2026-03,Unknown,61.00000



--- 2. Number and GBP Value of Reversals per Month ---


,month,reversal_count,total_reversal_value_gbp
0,2026-03,1,-6.2



--- 3. Foreign Currency Spend by Card per Month (in GBP) ---


,month,card_id,segment,foreign_spend_gbp
0,2026-02,C1004,Premium,178.75200
1,2026-03,C1003,Premium,33.86162
2,2026-03,C1004,Premium,102.03600
3,2026-03,C1005,Standard,12.04875



Gold tables have been successfully exported to CSV files:
- gold_segment_monthly_spend.csv
- gold_monthly_reversals.csv
- gold_foreign_spend_by_card.csv
